# PaliGemma QLoRA Fine-tuning for Remote Sensing
This notebook adapts the PaliGemma-3B model for remote sensing scene understanding using the RSICD dataset. It is optimized to run on a free Google Colab T4 GPU instance.

In [ ]:
!pip install -q -U transformers peft bitsandbytes datasets accelerate trl matplotlib

In [ ]:
import torch
from datasets import load_dataset
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training

# 1. Load Dataset
print('Loading arampacha/rsicd dataset...')
dataset = load_dataset('arampacha/rsicd', split='train[:10%]')
print(f"Loaded {len(dataset)} training samples.")

In [ ]:
# 2. Model & Processor Setup (4-bit Quantization)
model_id = 'google/paligemma-3b-pt-224'
processor = PaliGemmaProcessor.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=['q_proj', 'v_proj'], bias='none', task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 3. Training Loop
def collate_fn(examples):
    images = [example['image'].convert('RGB') for example in examples]
    texts = ['caption: ' + example['captions'][0] for example in examples]
    tokens = processor(text=texts, images=images, return_tensors='pt', padding='longest', truncation=True)
    labels = tokens['input_ids'].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    tokens['labels'] = labels
    return tokens

training_args = TrainingArguments(
    output_dir='./paligemma_results',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    max_steps=300,
    logging_steps=10,
    save_steps=100,
    optim='paged_adamw_8bit',
    fp16=True,
    remove_unused_columns=False,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=collate_fn,
)

print("Starting QLoRA Fine-tuning...")
trainer.train()

In [ ]:
# 4. Save and Export Weights
adapter_path = './paligemma_rs_lora'
model.save_pretrained(adapter_path)
processor.save_pretrained(adapter_path)

!zip -r paligemma_rs_lora.zip ./paligemma_rs_lora
print('Adapter saved and zipped! You can now download paligemma_rs_lora.zip from the sidebar.')